# Medidas de localización y variabilidad

Se analizan `edad_primer_union` y `num_hijos` con Polars. La media ponderada se calcula con NumPy y `factor_expansion`. Para la edad se utilizan solo los casos válidos; no se imputó debido a su alta proporción de faltantes.

In [1]:
from pathlib import Path
import sys

PROYECTO = Path.cwd().resolve()
if PROYECTO.name == 'notebooks':
    PROYECTO = PROYECTO.parent
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))

import polars as pl
import polars.selectors as cs
from src.analysis.medidas_descriptivas import (
    cargar_datos, comparacion_por_violencia, medidas_localizacion,
    medidas_variabilidad, validar_resultados
)

df = cargar_datos()
localizacion = medidas_localizacion(df)
variabilidad = medidas_variabilidad(df)
comparacion = comparacion_por_violencia(df)
validar_resultados(localizacion, variabilidad, comparacion)
print(f'Polars {pl.__version__}; registros: {df.height:,}')

Polars 1.44.2; registros: 105,715


## Medidas de localización

Se reportan media simple, media ponderada, mediana, todas las modas empatadas y los percentiles solicitados.

In [2]:
localizacion.with_columns(cs.float().round(2))

variable,n_validos,media_simple,media_ponderada,mediana_q2,moda,p10,q1,q3,p90
str,i64,f64,f64,f64,str,f64,f64,f64,f64
"""edad_primer_union""",9951,19.97,19.74,18.0,"""10""",10.0,12.0,25.0,34.0
"""num_hijos""",105715,2.06,1.85,3.0,"""3""",0.0,1.0,3.0,4.0


La edad de primera unión tiene media simple de **19.97 años** y media ponderada de **19.74 años**. La diferencia de 0.23 años aparece porque los registros tienen distinto factor de expansión; para describir la población nacional es preferible la media ponderada. La mediana es 18 años y el 50% central se encuentra entre 12 y 25 años. Estos resultados se basan en 9,951 casos válidos y no deben generalizarse sin reconocer la alta ausencia de datos.

Para `num_hijos`, la media simple es 2.06 y la ponderada 1.85. Su interpretación debe mencionar que 19,231 valores ausentes fueron imputados como cero conforme a la regla documentada.

## Medidas de variabilidad

La varianza y la desviación estándar son muestrales (`ddof=1`). El coeficiente de variación se calcula como $CV=s/\bar{x}\times100$ y el rango intercuartílico como $IQR=Q_3-Q_1$.

In [3]:
variabilidad.with_columns(cs.float().round(2))

variable,n_validos,minimo,maximo,rango,varianza_muestral,desviacion_estandar,coeficiente_variacion_pct,iqr
str,i64,f64,f64,f64,f64,f64,f64,f64
"""edad_primer_union""",9951,9.0,76.0,67.0,94.02,9.7,48.56,13.0
"""num_hijos""",105715,0.0,4.0,4.0,2.1,1.45,70.36,2.0


La edad presenta desviación estándar de 9.70 años, CV de 48.56% e IQR de 13 años. Esto indica una dispersión considerable respecto a su media y confirma que un único promedio no resume adecuadamente toda la distribución. El rango completo es sensible a valores extremos, por lo que el IQR ofrece una lectura más robusta.

## Comparación según reporte de violencia de pareja

In [4]:
comparacion.with_columns(cs.float().round(2))

variable,reporto_violencia,n_validos,media_simple,media_ponderada,mediana,rango,varianza_muestral,desviacion_estandar,coeficiente_variacion_pct,iqr
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""edad_primer_union""","""No""",6365,20.1,19.95,18.0,67.0,96.72,9.83,48.92,13.0
"""edad_primer_union""","""Sí""",3586,19.73,19.36,18.0,63.0,89.17,9.44,47.87,13.0
"""num_hijos""","""No""",86516,2.01,1.8,3.0,4.0,2.15,1.46,72.74,2.0
"""num_hijos""","""Sí""",19199,2.27,2.12,3.0,4.0,1.84,1.36,59.89,2.0


Entre los casos con edad válida, ambos grupos tienen mediana de 18 e IQR de 13 años. El CV fue **48.92%** entre quienes no reportaron violencia y **47.87%** entre quienes sí reportaron. Por tanto, en este conjunto no se observa el CV considerablemente mayor que plantea el cuestionario como supuesto hipotético.

Las diferencias son descriptivas y pequeñas. No demuestran que la edad de primera unión cause o prevenga violencia. Además, la elevada falta de edades puede introducir sesgo de selección. Para investigar una hipótesis explicativa convendría segmentar por `estado_civil_desc`, `nivel_escolaridad` y entidad, manteniendo el factor de expansión.